# ♻️ Waste Classification Model — Colab Training Launcher

This notebook contains **no application logic**.
Its only responsibility is to:
1. Mount Google Drive (checkpoints are saved there automatically)
2. Clone / pull the latest code from GitHub
3. Install dependencies
4. Verify the dataset
5. **Resume** from the latest Drive checkpoint — or start fresh
6. Run inference on real-world images (optional)
7. Download outputs (optional)

All source code lives in the repository. This notebook is simply a free GPU runtime.

---
**Repository:** `https://github.com/Shakir5665/Waste-Classification-Model.git`  
**Runtime:** Runtime → Change runtime type → **T4 GPU** (recommended)

---
### How resume works
After every epoch the training code automatically copies two things to your Drive:
- `WasteClassifier/checkpoints/epoch_NN_valloss_X.keras` — one file per epoch (never deleted)
- `WasteClassifier/best_model/best_model.keras` — overwritten every time val_loss improves

When your Colab session disconnects and you re-open this notebook:
- Run **Steps 1 → 4** again (Drive mount, clone/pull, install, dataset check)
- Then run **Step 5b** (Resume) instead of Step 5a (Fresh)

The script will scan the Drive checkpoints folder, find the highest-epoch file,
copy it to the local Colab disk, reload the model weights, and continue training
from exactly where it left off.


## Step 1 — Mount Google Drive

Drive is where checkpoints are saved after every epoch.  
Mount it **first** so the training script can write to it immediately.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/WasteClassifier'
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/best_model',  exist_ok=True)
print(f'Drive directories ready:')
print(f'  Checkpoints : {DRIVE_ROOT}/checkpoints')
print(f'  Best model  : {DRIVE_ROOT}/best_model')

## Step 2 — Clone / Pull the Repository

Run this cell once per Colab session.  
If the repo already exists it pulls the latest changes instead.


In [ ]:
import os

REPO_URL  = 'https://github.com/Shakir5665/Waste-Classification-Model.git'
REPO_NAME = 'Waste-Classification-Model'
REPO_DIR  = f'/content/{REPO_NAME}'

if os.path.exists(REPO_DIR):
    print('Repository already cloned. Pulling latest changes...')
    %cd {REPO_DIR}
    !git pull
else:
    print('Cloning repository...')
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

print(f'\nWorking directory: {os.getcwd()}')

## Step 3 — Install Dependencies


In [ ]:
!pip install -r requirements.txt -q
print('Dependencies installed.')

## Step 4 — Verify the Dataset

The dataset lives in `dataset/` inside the cloned repository.  
This cell confirms the expected folders are present.


In [ ]:
import os

dataset_path = os.path.join(REPO_DIR, 'dataset')

if os.path.exists(dataset_path):
    print(f'Dataset found at: {dataset_path}')
    for split in ['train', 'val', 'test']:
        split_path = os.path.join(dataset_path, split)
        if os.path.exists(split_path):
            classes = os.listdir(split_path)
            total   = sum(len(os.listdir(os.path.join(split_path, c)))
                          for c in classes if os.path.isdir(os.path.join(split_path, c)))
            print(f'  {split:5s} : {classes}  ({total} images)')
        else:
            print(f'  {split:5s} : NOT FOUND')
else:
    print('ERROR: dataset/ folder not found. Check that the folder exists in the repository.')

## Step 5 — Check Drive for Existing Checkpoints

Run this cell to see what is already saved on Drive before deciding whether to start fresh or resume.


In [ ]:
import os, re

ckpt_dir  = '/content/drive/MyDrive/WasteClassifier/checkpoints'
best_dir  = '/content/drive/MyDrive/WasteClassifier/best_model'

# --- epoch checkpoints ---
pattern = re.compile(r'epoch_(\d+)_valloss_([\d.]+)\.keras$')
ckpts   = sorted(
    [(int(m.group(1)), f) for f in os.listdir(ckpt_dir)
     if (m := pattern.match(f))],
    key=lambda x: x[0]
) if os.path.exists(ckpt_dir) else []

if ckpts:
    print(f'Found {len(ckpts)} epoch checkpoint(s) on Drive:')
    for epoch_num, fname in ckpts:
        marker = '  <-- LATEST' if epoch_num == ckpts[-1][0] else ''
        print(f'  Epoch {epoch_num:>3} : {fname}{marker}')
    print(f'\nWill resume from epoch {ckpts[-1][0] + 1} if you run Step 6b.')
else:
    print('No epoch checkpoints found on Drive — run Step 6a for a fresh start.')

# --- best model ---
best_file = os.path.join(best_dir, 'best_model.keras')
if os.path.exists(best_file):
    size_mb = os.path.getsize(best_file) / 1_048_576
    print(f'\nbest_model.keras found on Drive  ({size_mb:.1f} MB)')
else:
    print('\nbest_model.keras not yet on Drive.')

## Step 6a — Train from Scratch  *(run this OR Step 6b, not both)*

Use this when there are **no Drive checkpoints** yet, or when you deliberately want to restart training.

After every epoch the script will:
- Save `epoch_NN_valloss_X.keras` locally **and** copy it to Drive
- Save / overwrite `best_model.keras` locally **and** on Drive (only when val_loss improves)


In [ ]:
!python scripts/train.py

## Step 6b — Resume Training  *(run this OR Step 6a, not both)*

Use this **after a Colab disconnect** when Drive checkpoints already exist.

What happens automatically:
1. Scans `WasteClassifier/checkpoints/` on Drive for all `epoch_NN_valloss_X.keras` files
2. Picks the one with the **highest epoch number**
3. Copies it to the local Colab disk and loads the model weights
4. Restores `best_model.keras` from Drive to local disk (if missing locally)
5. Calls `model.fit(..., initial_epoch=N)` so Keras continues from epoch N+1
6. All Drive sync callbacks remain active — new checkpoints keep being saved


In [ ]:
!python scripts/train.py --resume

## Step 7 — (Optional) Run Inference on Real-World Images

Classifies the images in `Realworld_data/` using the trained model.


In [ ]:
!python scripts/predict.py --input Realworld_data/

## Step 8 — (Optional) Download Outputs

Download the trained model and evaluation reports directly to your browser.  
The best model is also already saved on your Drive — no download needed for that one.


In [ ]:
from google.colab import files
import os

outputs = [
    'outputs/models/waste_classifier.keras',
    'outputs/reports/training_curves.png',
    'outputs/reports/confusion_matrix.png',
    'outputs/reports/confusion_matrix_normalised.png',
    'outputs/reports/roc_curve.png',
    'outputs/reports/precision_recall_curve.png',
    'outputs/reports/per_class_metrics.png',
    'outputs/reports/classification_report.txt',
    'outputs/reports/training_summary.json',
]

for path in outputs:
    if os.path.exists(path):
        files.download(path)
        print(f'Downloaded: {path}')
    else:
        print(f'Not found (skipped): {path}')

## Step 9 — (Optional) Inspect Drive Checkpoint Status

Re-run at any time to see what is currently saved on Drive.


In [ ]:
import os, re

ckpt_dir = '/content/drive/MyDrive/WasteClassifier/checkpoints'
best_dir = '/content/drive/MyDrive/WasteClassifier/best_model'

print('=' * 55)
print('  Google Drive — Checkpoint Status')
print('=' * 55)

pattern = re.compile(r'epoch_(\d+)_valloss_([\d.]+)\.keras$')
ckpts   = sorted(
    [(int(m.group(1)), float(m.group(2)), f)
     for f in os.listdir(ckpt_dir)
     if (m := pattern.match(f))],
    key=lambda x: x[0]
) if os.path.exists(ckpt_dir) else []

if ckpts:
    print(f'\n  Epoch checkpoints ({len(ckpts)} total):')
    for epoch_num, val_loss, fname in ckpts:
        print(f'    Epoch {epoch_num:>3}  val_loss={val_loss:.4f}  {fname}')
    best_ep, best_loss, _ = min(ckpts, key=lambda x: x[1])
    last_ep = ckpts[-1][0]
    print(f'\n  Last epoch saved : {last_ep}')
    print(f'  Best val_loss    : {best_loss:.4f}  (epoch {best_ep})')
else:
    print('\n  No epoch checkpoints found.')

best_file = os.path.join(best_dir, 'best_model.keras')
if os.path.exists(best_file):
    size_mb = os.path.getsize(best_file) / 1_048_576
    print(f'\n  best_model.keras : {best_file}  ({size_mb:.1f} MB)')
else:
    print('\n  best_model.keras : not yet saved on Drive')

print('\n' + '=' * 55)